# 03 — Country, Category, and Position Analysis


> **Post-shared-task analysis.** Gold test labels were public when this analysis was
> designed. Nothing in this notebook changes the official CI=0.035 submission or the
> third-place ranking. Results are retrospective and must not be described as untouched
> test-set estimates.

This notebook analyzes the best official submission (QLoRA 2,600) by released
metadata and true-statement position. Wilson intervals are descriptive; overlapping
groups and small denominators should not be interpreted as country effects.

In [ ]:
from pathlib import Path
from collections import Counter
import csv, io, json, os, urllib.request, zipfile

import numpy as np
import pandas as pd


def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'README.md').exists() and (candidate / 'Test').exists():
            return candidate.resolve()
    raise FileNotFoundError('Run this notebook from the IE2026-HalDetect repository.')


ROOT = find_repo_root()
HERE = ROOT / 'post_task_analysis'
CACHE = HERE / 'cache'
OUTPUT = HERE / 'outputs'
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

GOLD_URL = (
    'https://huggingface.co/datasets/QCRI/ImageEval-ArabicNLP26/'
    'resolve/main/task1b/test_en.jsonl'
)
gold_override = os.getenv('IMAGEEVAL_TEST_GOLD')
GOLD_PATH = Path(gold_override) if gold_override else CACHE / 'test_en.jsonl'
if not GOLD_PATH.exists():
    print('Downloading released gold test JSONL...')
    urllib.request.urlretrieve(GOLD_URL, GOLD_PATH)


def read_gold(path=GOLD_PATH):
    rows = [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]
    assert len(rows) == 1000, f'Expected 1,000 gold items, found {len(rows)}'
    frame = pd.DataFrame(rows)
    assert frame['id'].is_unique
    assert frame['labels'].map(lambda x: len(x) == 3 and sum(x) == 1).all()
    frame['gold_idx'] = frame['labels'].map(lambda x: x.index(True))
    return frame


def load_prediction_zip(path, gold):
    path = Path(path)
    assert path.exists(), path
    with zipfile.ZipFile(path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
        assert len(csv_names) == 1, (path, csv_names)
        with io.TextIOWrapper(archive.open(csv_names[0]), encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))
    raw = pd.DataFrame(rows)
    required = {'id', 'statement_index', 'prediction'}
    assert required.issubset(raw.columns), (path, raw.columns)
    raw['statement_index'] = raw['statement_index'].astype(int)
    raw['pred_bool'] = raw['prediction'].str.strip().str.lower().map(
        {'true': True, 'false': False})
    assert raw['pred_bool'].notna().all(), f'Unparseable prediction in {path}'
    assert not raw.duplicated(['id', 'statement_index']).any()
    assert set(raw['statement_index']) == {0, 1, 2}
    grouped = raw.sort_values(['id', 'statement_index']).groupby('id', sort=False)
    vectors = grouped['pred_bool'].apply(list)
    assert vectors.map(len).eq(3).all()
    assert set(vectors.index) == set(gold['id']), f'ID mismatch in {path}'

    result = gold[['id', 'gold_idx']].copy()
    by_id = vectors.to_dict()
    result['pred_vector'] = result['id'].map(by_id)
    result['format_valid'] = result['pred_vector'].map(lambda x: sum(x) == 1)
    result['pred_idx'] = result['pred_vector'].map(
        lambda x: x.index(True) if sum(x) == 1 else np.nan)
    result['correct'] = result['format_valid'] & result['pred_idx'].eq(result['gold_idx'])
    result['error'] = ~result['correct']
    return result


SUBMISSIONS = {
    'Elimination': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-elimination/prediction_en.zip',
    'Socratic': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-socratic/prediction_en.zip',
    'Devils advocate': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-devils-advocate/prediction_en.zip',
    'Evidence first': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-evidence-first/prediction_en.zip',
    'Attribute checklist': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-attribute-checklist/prediction_en.zip',
    'Confidence ranked': ROOT / 'Test/qwen2p5-3b-7b/All COT variations/cot-confidence-ranked/prediction_en.zip',
    'QLoRA 2,000': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2k-image/prediction_en.zip',
    'QLoRA 2,348': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p3k-image/prediction_en.zip',
    'QLoRA 2,600': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-2p6k-image/prediction_en.zip',
    'QLoRA 3,000 legacy': ROOT / 'Test/qwen2p5-3b-7b/qlora-q7b-3k-image/prediction_en.zip',
}

gold = read_gold()
predictions = {name: load_prediction_zip(path, gold) for name, path in SUBMISSIONS.items()}
summary = pd.DataFrame([
    {
        'system': name,
        'n': len(frame),
        'errors': int(frame['error'].sum()),
        'CI': frame['error'].mean(),
        'accuracy': frame['correct'].mean(),
        'format_failures': int((~frame['format_valid']).sum()),
    }
    for name, frame in predictions.items()
]).sort_values(['CI', 'system']).reset_index(drop=True)
display(summary)

In [ ]:
import math

SYSTEM = 'QLoRA 2,600'
evaluated = gold.merge(
    predictions[SYSTEM][['id', 'pred_idx', 'correct', 'error']], on='id')


def wilson_interval(errors, total, z=1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    p = errors / total
    denominator = 1 + z*z/total
    center = (p + z*z/(2*total)) / denominator
    margin = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
    return center - margin, center + margin


def grouped_errors(field):
    rows = []
    for value, frame in evaluated.groupby(field, dropna=False):
        n = len(frame)
        errors = int(frame['error'].sum())
        low, high = wilson_interval(errors, n)
        rows.append({
            'field': field, 'group': value, 'n': n, 'errors': errors,
            'error_rate': errors/n, 'wilson_95_low': low, 'wilson_95_high': high,
        })
    return pd.DataFrame(rows).sort_values(['error_rate', 'errors'], ascending=False)


subgroup = pd.concat([
    grouped_errors('country'), grouped_errors('category'), grouped_errors('subcategory')
], ignore_index=True)
subgroup.to_csv(OUTPUT / 'subgroup_error_intervals.csv', index=False)
display(subgroup.query("field == 'country'").head(15))
display(subgroup.query("field == 'category'"))
display(subgroup.query("field == 'subcategory'").head(15))

## Position-specific error and confusion

In [ ]:
position_rows = []
for position, frame in evaluated.groupby('gold_idx'):
    n = len(frame)
    errors = int(frame['error'].sum())
    low, high = wilson_interval(errors, n)
    position_rows.append({
        'gold_position_1based': int(position + 1),
        'n': n, 'errors': errors, 'error_rate': errors/n,
        'wilson_95_low': low, 'wilson_95_high': high,
    })
position_rates = pd.DataFrame(position_rows)
position_rates.to_csv(OUTPUT / 'position_error_intervals.csv', index=False)
display(position_rates)

confusion = pd.crosstab(
    evaluated['gold_idx'] + 1,
    evaluated['pred_idx'] + 1,
    rownames=['gold position'], colnames=['predicted position'], dropna=False)
confusion.to_csv(OUTPUT / 'position_confusion_matrix.csv')
display(confusion)

import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(4.8, 4))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues')
plt.title(f'{SYSTEM}: true-position confusion')
plt.tight_layout()
plt.savefig(OUTPUT / 'position_confusion_heatmap.png', dpi=180)
plt.show()

## Export the 35 errors for qualitative inspection

In [ ]:
errors = evaluated.loc[evaluated['error']].copy()
errors['statements_json'] = errors['statements'].map(json.dumps)
errors.drop(columns=['labels', 'statements']).to_csv(
    OUTPUT / 'qlora_2600_test_errors.csv', index=False)
print(f'Exported {len(errors)} errors; expected 35.')
assert len(errors) == 35